In [1]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import timm



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False




In [3]:
def crop_fundus_border_pil(img: Image.Image, tol: int = 7) -> Image.Image:
    """
    Minimal, deterministic APTOS-style border crop:
    removes near-black background so models see the fundus region instead of padded borders.
    This preserves the same downstream model/inference logic while improving score alignment.
    """
    if img.mode != "RGB":
        img = img.convert("RGB")
    arr = np.asarray(img)
    gray = (0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]).astype(
        np.float32
    )
    mask = gray > float(tol)

    if not mask.any():
        return img  # fallback: nothing to crop

    ys, xs = np.where(mask)
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1

    if (y1 - y0) < 10 or (x1 - x0) < 10:
        return img

    return img.crop((x0, y0, x1, y1))




In [4]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + ".png")
        image = Image.open(img_name).convert("RGB")

        image = crop_fundus_border_pil(image, tol=7)

        if self.transform:
            image = self.transform(image)

        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [5]:
model_paths = {
    "resnet18": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
    "efficientnet_b5": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/efficientnet_b5.pth",
    "inception_resnet_v2": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_resnet_v2.pth",
    "inception_v4": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_v4.pth",
    "seresnext50_32x4d": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext50_32x4d.pth",
    "seresnext101_32x4d": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext101_32x4d.pth",
}

model_names = {
    "resnet18": "resnet18",
    "efficientnet_b5": "efficientnet_b5",
    "inception_resnet_v2": "inception_resnet_v2",
    "inception_v4": "inception_v4",
    "seresnext50_32x4d": "seresnext50_32x4d",
    "seresnext101_32x4d": "seresnext101_32x4d",
}



In [6]:
validation_scores = {
    "resnet18": 0.887,
    "efficientnet_b5": 0.952,
    "inception_resnet_v2": 0.822,
    "inception_v4": 0.888,
    "seresnext50_32x4d": 0.709,
    "seresnext101_32x4d": 0.951,
}



In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_list = []
active_model_keys = []

for model_key, ckpt_path in model_paths.items():
    model_name = model_names[model_key]
    has_ckpt = os.path.exists(ckpt_path)

    try:
        model = timm.create_model(model_name, pretrained=not has_ckpt, num_classes=5)
    except Exception as e:
        print(
            f"[WARN] Skipping model {model_key} ({model_name}) due to create_model error: {e}"
        )
        continue

    if has_ckpt:
        try:
            state = torch.load(ckpt_path, map_location="cpu")
            if (
                isinstance(state, dict)
                and "state_dict" in state
                and isinstance(state["state_dict"], dict)
            ):
                state = state["state_dict"]
            model.load_state_dict(state, strict=True)
        except Exception as e:
            print(
                f"[WARN] Skipping model {model_key} due to checkpoint load error: {e}"
            )
            continue

    model.to(device)
    model.eval()

    models_list.append(model)
    active_model_keys.append(model_key)

if len(models_list) == 0:
    raise RuntimeError(
        "No models could be created/loaded. Check dataset availability and timm model names."
    )

print("Active models:", active_model_keys)



model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/224M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/171M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/111M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/196M [00:00<?, ?B/s]

Active models: ['resnet18', 'efficientnet_b5', 'inception_resnet_v2', 'inception_v4', 'seresnext50_32x4d', 'seresnext101_32x4d']


In [8]:
active_scores = {k: validation_scores[k] for k in active_model_keys}
total_score = float(sum(active_scores.values()))
if total_score <= 0:
    weights = {k: 1.0 / len(active_model_keys) for k in active_model_keys}
else:
    weights = {k: (v / total_score) for k, v in active_scores.items()}

s = float(sum(weights.values()))
weights = {k: (v / s) for k, v in weights.items()}

print("Ensemble weights:", weights)



Ensemble weights: {'resnet18': 0.17028220387790363, 'efficientnet_b5': 0.18276060664234978, 'inception_resnet_v2': 0.1578038011134575, 'inception_v4': 0.17047417930504896, 'seresnext50_32x4d': 0.1361105778460357, 'seresnext101_32x4d': 0.18256863121520445}


In [9]:
# Change (score-relevant, minimal): use per-model timm eval transforms so each backbone
# receives the input size / normalization it expects. This keeps ensemble+severity logic
# identical, but fixes a major preprocessing mismatch that can produce very poor QWK.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

per_model_cfg = {}
per_model_transform = {}
per_model_size = {}

for model_key, model in zip(active_model_keys, models_list):
    cfg = timm.data.resolve_model_data_config(model)
    # keep mean/std fixed ImageNet for stability across models (as before),
    # but keep model-expected spatial sizing/cropping.
    cfg = dict(cfg)
    cfg["mean"] = IMAGENET_MEAN
    cfg["std"] = IMAGENET_STD
    # ensure interpolation is deterministic and supported
    cfg["interpolation"] = "bilinear"

    per_model_cfg[model_key] = cfg
    inp = cfg.get("input_size", (3, 224, 224))
    per_model_size[model_key] = int(inp[-1])

    # timm expects input_size as (C,H,W) for create_transform; pass resolved cfg directly.
    per_model_transform[model_key] = timm.data.create_transform(
        **cfg, is_training=False
    )

print("Per-model input sizes:", per_model_size)
print("Using fixed mean/std:", IMAGENET_MEAN, IMAGENET_STD)




Per-model input sizes: {'resnet18': 224, 'efficientnet_b5': 448, 'inception_resnet_v2': 299, 'inception_v4': 299, 'seresnext50_32x4d': 224, 'seresnext101_32x4d': 224}
Using fixed mean/std: (0.485, 0.456, 0.406) (0.229, 0.224, 0.225)


In [10]:
def _seed_worker(worker_id: int):
    worker_seed = (SEED + worker_id) % (2**32 - 1)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


g = torch.Generator()
g.manual_seed(SEED)

test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"
test_dataset = BlindnessDataset(test_csv_file, test_root_dir, transform=None, test=True)

use_pin = bool(torch.cuda.is_available())

test_loader = DataLoader(
    test_dataset,
    batch_size=8,  # keep safe memory since per-model transform happens per-batch
    shuffle=False,
    num_workers=2,
    pin_memory=use_pin,
    worker_init_fn=_seed_worker,
    generator=g,
)




In [11]:
def predict_expected_severity(loader: DataLoader) -> np.ndarray:
    all_outputs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Infer"):
            if isinstance(batch, (tuple, list)):
                images = batch[0]
            else:
                images = batch

            # images are PILs? no, dataset returns PIL when transform=None; DataLoader will
            # collate into a list. Handle that deterministically.
            if isinstance(images, list):
                pil_list = images
            else:
                # if user later sets a tensor transform, keep compatibility
                pil_list = None

            outputs = []
            for model_key, model in zip(active_model_keys, models_list):
                if pil_list is not None:
                    tfm = per_model_transform[model_key]
                    imgs_t = torch.stack([tfm(im) for im in pil_list], dim=0)
                else:
                    # already a tensor batch; assume correct shape
                    imgs_t = images

                imgs_t = imgs_t.to(device, non_blocking=use_pin)

                logits = model(imgs_t)
                probs = nn.functional.softmax(logits, dim=1)
                outputs.append(weights[model_key] * probs)

            weighted_outputs = torch.stack(outputs, dim=0).sum(dim=0)
            all_outputs.append(weighted_outputs.cpu().numpy())

    all_outputs = np.concatenate(all_outputs, axis=0)
    class_values = np.arange(5, dtype=np.float32)
    severity = (all_outputs * class_values[None, :]).sum(axis=1)  # in [0,4]
    return severity




In [12]:
def qwk(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 5) -> float:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    assert y_true.shape == y_pred.shape

    O = np.zeros((n_classes, n_classes), dtype=np.float64)
    for a, b in zip(y_true, y_pred):
        if 0 <= a < n_classes and 0 <= b < n_classes:
            O[a, b] += 1.0

    act_hist = O.sum(axis=1)
    pred_hist = O.sum(axis=0)
    E = np.outer(act_hist, pred_hist)
    if E.sum() > 0:
        E = E * (O.sum() / E.sum())

    W = np.zeros((n_classes, n_classes), dtype=np.float64)
    for i in range(n_classes):
        for j in range(n_classes):
            W[i, j] = ((i - j) ** 2) / ((n_classes - 1) ** 2)

    num = (W * O).sum()
    den = (W * E).sum()
    if den == 0:
        return 0.0
    return 1.0 - num / den


def apply_thresholds(severity: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    preds = np.digitize(severity, thresholds, right=False).astype(int)
    return np.clip(preds, 0, 4)


def make_stratified_folds(
    y: np.ndarray, n_splits: int = 5, seed: int = 42
) -> np.ndarray:
    rng = np.random.RandomState(seed)
    y = y.astype(int)
    fold_id = np.empty(len(y), dtype=int)
    for cls in range(5):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        parts = np.array_split(idx, n_splits)
        for f in range(n_splits):
            fold_id[parts[f]] = f
    return fold_id


def optimize_thresholds_1d(
    sev: np.ndarray, y: np.ndarray, init_thr: np.ndarray
) -> np.ndarray:
    thr = init_thr.astype(np.float64).copy()
    thr.sort()

    sev_min, sev_max = float(np.min(sev)), float(np.max(sev))
    grid = np.linspace(sev_min, sev_max, 320, dtype=np.float64)

    best_pred = apply_thresholds(sev, thr)
    best_score = qwk(y, best_pred)

    for _ in range(2):  # keep same passes for minimal change
        for i in range(4):
            lo = sev_min if i == 0 else thr[i - 1] + 1e-6
            hi = sev_max if i == 3 else thr[i + 1] - 1e-6
            candidates = grid[(grid > lo) & (grid < hi)]
            if candidates.size == 0:
                continue

            local_best_thr = thr[i]
            local_best_score = best_score

            for c in candidates:
                tmp = thr.copy()
                tmp[i] = c
                tmp.sort()
                pred = apply_thresholds(sev, tmp)
                sc = qwk(y, pred)
                if sc > local_best_score + 1e-12:
                    local_best_score = sc
                    local_best_thr = c

            thr[i] = local_best_thr
            thr.sort()
            best_score = local_best_score

    return thr




In [13]:
train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
train_root_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"

train_df = pd.read_csv(train_csv_file)
y_train = train_df["diagnosis"].astype(int).values

fold_id = make_stratified_folds(y_train, n_splits=5, seed=SEED)

train_dataset = BlindnessDataset(
    train_csv_file, train_root_dir, transform=None, test=False
)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,  # match test loader for consistent preprocessing path
    shuffle=False,
    num_workers=2,
    pin_memory=use_pin,
    worker_init_fn=_seed_worker,
    generator=g,
)

train_severity = predict_expected_severity(train_loader).astype(np.float32)
if len(train_severity) != len(train_df):
    raise RuntimeError("Train severity length mismatch.")

oof_pred_foldthr = np.zeros(len(train_df), dtype=int)
fold_thresholds = []

for f in range(5):
    tr_idx = np.where(fold_id != f)[0]
    va_idx = np.where(fold_id == f)[0]

    sev_tr = train_severity[tr_idx]
    y_tr = y_train[tr_idx]

    counts = np.bincount(y_tr, minlength=5).astype(np.float64)
    probs = counts / counts.sum()
    cum = np.cumsum(probs)
    q = np.clip(cum[:4], 1e-6, 1 - 1e-6)
    init_thresholds = np.quantile(sev_tr, q)
    init_thresholds = np.maximum.accumulate(init_thresholds)

    thr_f = optimize_thresholds_1d(sev_tr, y_tr, init_thresholds)
    thr_f = np.maximum.accumulate(thr_f)
    fold_thresholds.append(thr_f)

    oof_pred_foldthr[va_idx] = apply_thresholds(train_severity[va_idx], thr_f)

fold_thresholds = np.vstack(fold_thresholds)
avg_thresholds = np.maximum.accumulate(fold_thresholds.mean(axis=0))

print(
    "OOF QWK (using fold-specific thresholds on their held-out fold):",
    qwk(y_train, oof_pred_foldthr),
)
print("Fold thresholds:\n", fold_thresholds)
print("Averaged thresholds (diagnostic):", avg_thresholds)
print("OOF pred distribution (fold-thr):", np.bincount(oof_pred_foldthr, minlength=5))

counts_full = np.bincount(y_train, minlength=5).astype(np.float64)
probs_full = counts_full / counts_full.sum()
cum_full = np.cumsum(probs_full)
q_full = np.clip(cum_full[:4], 1e-6, 1 - 1e-6)
init_thresholds_full = np.quantile(train_severity, q_full)
init_thresholds_full = np.maximum.accumulate(init_thresholds_full)

thresholds = optimize_thresholds_1d(train_severity, y_train, init_thresholds_full)
thresholds = np.maximum.accumulate(thresholds)

oof_pred_finalthr = apply_thresholds(train_severity, thresholds)
print(
    "OOF QWK (single thresholds fit on full OOF severities):",
    qwk(y_train, oof_pred_finalthr),
)
print("Final thresholds used for test:", thresholds)
print("OOF pred distribution (final-thr):", np.bincount(oof_pred_finalthr, minlength=5))



Infer:   0%|          | 0/412 [00:01<?, ?it/s]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/collate.py", line 211, in collate
    return [
           ^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/collate.py", line 212, in <listcomp>
    collate(samples, collate_fn_ma

In [14]:
test_severity = predict_expected_severity(test_loader)
final_predictions = apply_thresholds(test_severity, thresholds)
final_predictions = np.clip(final_predictions, 0, 4).astype(np.int64)

uniq_vals = np.unique(final_predictions)
if uniq_vals.size == 1:
    print(
        f"[WARN] Predictions collapsed to a single class ({int(uniq_vals[0])}). "
        "Applying deterministic fallback thresholds based on train label quantiles."
    )

    counts_full = np.bincount(y_train, minlength=5).astype(np.float64)
    probs_full = counts_full / counts_full.sum()
    cum_full = np.cumsum(probs_full)
    q_full = np.clip(cum_full[:4], 1e-6, 1 - 1e-6)

    thr_fallback = np.quantile(train_severity.astype(np.float64), q_full)
    thr_fallback = np.maximum.accumulate(thr_fallback).astype(np.float64)

    final_predictions = apply_thresholds(test_severity, thr_fallback).astype(np.int64)
    print("Fallback thresholds:", thr_fallback)
    print(
        "Post-fallback pred distribution:",
        np.bincount(final_predictions, minlength=5),
    )

final_predictions = np.clip(final_predictions, 0, 4).astype(np.int64)

sample_path = "/kaggle/input/aptos2019-blindness-detection/sample_submission.csv"
sample_df = pd.read_csv(sample_path)
sample_ids = sample_df["id_code"].astype(str).values

test_ids = pd.read_csv(test_csv_file)["id_code"].astype(str).values

if len(test_ids) != len(final_predictions):
    raise RuntimeError(
        f"Prediction length mismatch: {len(final_predictions)} vs {len(test_ids)}"
    )
if len(sample_ids) != len(final_predictions):
    raise RuntimeError(
        f"Sample submission length mismatch: {len(sample_ids)} vs {len(final_predictions)}"
    )
if len(np.unique(test_ids)) != len(test_ids):
    raise RuntimeError(
        "Duplicate id_code found in test.csv; submission would be ambiguous."
    )
if len(np.unique(sample_ids)) != len(sample_ids):
    raise RuntimeError(
        "Duplicate id_code found in sample_submission.csv; cannot guarantee ordering."
    )

pred_by_id = dict(zip(test_ids, final_predictions.tolist()))
missing = [i for i in sample_ids if i not in pred_by_id]
extra = [i for i in pred_by_id.keys() if i not in set(sample_ids)]
if len(missing) > 0:
    raise RuntimeError(
        f"Missing predictions for {len(missing)} ids (e.g. {missing[:5]})."
    )
if len(extra) > 0:
    raise RuntimeError(
        f"Found {len(extra)} unexpected ids not in sample_submission (e.g. {extra[:5]})."
    )

ordered_predictions = np.array([pred_by_id[i] for i in sample_ids], dtype=np.int64)
ordered_predictions = np.clip(ordered_predictions, 0, 4).astype(np.int64)

ordered_predictions_pyint = [int(x) for x in ordered_predictions.tolist()]

submission_df = pd.DataFrame(
    {"id_code": sample_ids, "diagnosis": ordered_predictions_pyint}
)

if not np.array_equal(submission_df["id_code"].astype(str).values, sample_ids):
    raise RuntimeError("Submission id_code order mismatch vs sample_submission.csv.")

valid_set = set(range(5))
vals = submission_df["diagnosis"].tolist()
if any((v not in valid_set) for v in vals):
    bad = sorted({v for v in vals if v not in valid_set})[:10]
    raise RuntimeError(f"Invalid diagnosis values found (not in 0..4). Examples: {bad}")

uniq_vals = sorted(set(vals))
if len(uniq_vals) == 1:
    print(
        f"[WARN] Submission predictions are constant ({uniq_vals[0]}). "
        "Writing submission.csv anyway to ensure a valid Kaggle submission."
    )

submission_df.to_csv("submission.csv", index=False)
submission_df.to_csv("submission_int.csv", index=False)

print(submission_df.head())
print("Pred distribution:", np.bincount(np.array(vals, dtype=int), minlength=5))
print("Wrote submission.csv with shape:", submission_df.shape)
print("Also wrote submission_int.csv (same content, explicitly Python-int labels).")
print("dtypes (pandas view):", submission_df.dtypes.to_dict())

Infer:   0%|          | 0/46 [00:01<?, ?it/s]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/collate.py", line 240, in collate
    raise TypeError(default_collate_err_msg_format.format(elem_type))
TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; foun